# Census Normalizer — walkthrough

Brokers send employee census files in every format imaginable. This notebook shows a messy file going in and a clean, validated, underwriting-ready dataset coming out, with the problem rows flagged.

In [ ]:
import sys; sys.path.insert(0, '../src')
import pandas as pd
from generate_synthetic_censuses import main as gen
from normalizer import normalize_file, map_columns
from validator import validate

gen()  # generates 4 synthetic broker files (fake data, no PHI)

Generated 4 synthetic census files in data/raw/:
  - broker_a_census.csv
  - broker_b_census.csv
  - broker_c_census.csv
  - broker_d_census.xlsx


## 1. Same data, four different shapes

In [2]:
for f in ['broker_a_census.csv','broker_b_census.csv','broker_c_census.csv']:
    df = pd.read_csv('../data/raw/'+f, dtype=str)
    print(f'\n{f}:'); print('  columns:', list(df.columns))
    print(df.head(2).to_string(index=False))


broker_a_census.csv:
  columns: ['EmpID', 'First Name', 'Last Name', 'DOB', 'Gender', 'Relationship', 'Zip', 'State']
EmpID First Name Last Name        DOB Gender Relationship   Zip State
E0001      Angel      Hill 1988-12-24      M           EE 29757    DC
E0002      Erica   Mcclain 1993-02-26      F           EE 11896    OR

broker_b_census.csv:
  columns: ['Employee Identifier', 'Member First Name', 'Member Last Name', 'Date of Birth', 'Sex', 'Relationship to Subscriber', 'Zip Code', 'Home State']
Employee Identifier Member First Name Member Last Name Date of Birth  Sex Relationship to Subscriber Zip Code Home State
              E0001            Meghan            Davis    06/01/2000 Male                   Employee    56019         MP
              E0001              Adam            Davis    09/06/2007 Male                      Child    56019         MP

broker_c_census.csv:
  columns: ['id', 'fname', 'lname', 'birth_date', 'gender', 'rel', 'zipcode', 'st']
   id  fname   lname bir

## 2. Column mapping — no per-broker code

Synonym dictionary + fuzzy similarity, so an unseen broker format still maps.

In [3]:
raw = pd.read_csv('../data/raw/broker_c_census.csv', dtype=str)
print('messy headers:', list(raw.columns))
for k,v in map_columns(raw).items():
    print(f'   {k:12} -> {v}')

messy headers: ['id', 'fname', 'lname', 'birth_date', 'gender', 'rel', 'zipcode', 'st']
   id           -> employee_id
   fname        -> first_name
   lname        -> last_name
   birth_date   -> date_of_birth
   gender       -> gender
   rel          -> relationship
   zipcode      -> zip_code
   st           -> state


## 3. Normalize + validate the messy file

Broker C has injected: duplicates, a missing DOB, a missing ZIP, mixed date formats, and an orphan dependent.

In [4]:
clean, mapping = normalize_file(raw, group_id='G003', source_file='broker_c_census.csv')
report, flagged = validate(clean)
print(f"Quality score: {report['quality_score']}/100")
print(f"Ready for underwriting: {report['ready_for_underwriting']}")
print(f"Rows: {report['rows_total']}  clean: {report['rows_clean']}  flagged: {report['rows_flagged']}\n")
for ck in report['checks']:
    print(f"  [{ck['severity']:>8}] {ck['check']}: {ck['count']} — {ck['message']}")

Quality score: 76.7/100
Ready for underwriting: False
Rows: 60  clean: 54  flagged: 6

  [ warning] duplicate_member: 2 — Exact duplicate member rows (same person appears more than once)
  [critical] missing_dob: 1 — Member missing date of birth -- cannot age-rate
  [critical] missing_zip: 1 — Member missing ZIP -- cannot area-rate
  [critical] orphan_dependent: 1 — Dependent has no matching employee/subscriber in the file
  [ warning] value_coercion_issue: 3 — Value needed cleaning or could not be fully standardized


## 4. The flagged rows — what a human should review

In [5]:
flagged_df = clean[clean['row_issues'].fillna('').str.len() > 0]
flagged_df[['employee_id','first_name','date_of_birth','gender','zip_code','relationship','row_issues']]

,employee_id,first_name,date_of_birth,gender,zip_code,relationship,row_issues
3,E0003,Katherine,1982-06-30,U,85160,employee,unknown gender 'nan' -> U
7,E0005,Joseph,1991-09-17,M,NaN,employee,missing zip_code
10,E0007,Justin,NaN,F,25944,employee,future DOB 2027-06-27


## 5. A clean broker file scores 100 and passes through

In [6]:
raw_a = pd.read_csv('../data/raw/broker_a_census.csv', dtype=str)
clean_a, _ = normalize_file(raw_a, group_id='G001')
rep_a, _ = validate(clean_a)
print(f"broker_a score: {rep_a['quality_score']}/100  ready: {rep_a['ready_for_underwriting']}")
clean_a.head()

broker_a score: 100.0/100  ready: True


,group_id,employee_id,first_name,last_name,date_of_birth,gender,relationship,zip_code,state,source_file,row_issues
0,G001,E0001,Angel,Hill,1988-12-24,M,employee,29757,DC,,
1,G001,E0002,Erica,Mcclain,1993-02-26,F,employee,11896,OR,,
2,G001,E0003,Anthony,Robinson,1979-08-18,M,employee,66738,PA,,
3,G001,E0003,James,Robinson,1962-08-17,F,spouse,66738,PA,,
4,G001,E0003,Bridget,Robinson,2006-03-21,M,child,66738,PA,,
